# Milestone 1: Data Profiling

Profil awal dataset besar Indonesia menggunakan Polars LazyFrame dan DuckDB. Isi sumber serta deskripsi dataset pada `data/README.md` sebelum menjalankan notebook ini.

In [10]:
from pathlib import Path

import duckdb
import polars as pl

PROJECT_ROOT = Path.cwd().resolve()
DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'SFinDSet'

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f'Dataset tidak ditemukan: {DATA_PATH}. '
        'Unduh data ke data/raw/ lalu sesuaikan DATA_PATH.'
    )

print(f'Dataset: {DATA_PATH.name}')
print(f'Ukuran file: {DATA_PATH.stat().st_size / 1_000_000:.1f} MB')

FileNotFoundError: Dataset tidak ditemukan: /home/jovyan/work/notebooks/data/raw/SFinDSet. Unduh data ke data/raw/ lalu sesuaikan DATA_PATH.

## Profil Polars

`scan_parquet` hanya membangun rencana query; file baru dibaca ketika `collect()` dipanggil.

In [11]:
lazy_data = pl.scan_parquet(DATA_PATH)
print(lazy_data.collect_schema())

ringkasan = lazy_data.select([
    pl.len().alias('jumlah_baris'),
    pl.all().null_count().name.suffix('_null'),
]).collect()
ringkasan

FileNotFoundError: No such file or directory (os error 2): /home/jovyan/work/notebooks/data/raw/SFinDSet

In [2]:
contoh_data = lazy_data.head(10).collect()
contoh_data

NameError: name 'lazy_data' is not defined

## Profil DuckDB

DuckDB membaca file Parquet langsung dari disk dan menghasilkan ringkasan statistik setiap kolom.

In [3]:
connection = duckdb.connect()
profil_duckdb = connection.execute(
    'SUMMARIZE SELECT * FROM read_parquet(?)', [str(DATA_PATH)]
).pl()
profil_duckdb

IOException: IO Error: No files found that match the pattern "/home/jovyan/work/notebooks/data/raw/dataset.parquet"

LINE 1: SUMMARIZE SELECT * FROM read_parquet(?)
                                ^

## Catatan Milestone 1

Tulis temuan awal di bawah ini: ukuran data, jumlah baris, tipe kolom, kolom dengan null terbanyak, serta anomali yang perlu ditangani pada Milestone 2.